In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost catboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import warnings
warnings.filterwarnings('ignore')

!apt-get install -y fonts-noto-cjk > /dev/null 2>&1
!rm ~/.cache/matplotlib -rf

from matplotlib import rcParams
rcParams['font.family'] = 'Noto Sans CJK JP'
rcParams['axes.unicode_minus'] = False

print("✅ 모든 라이브러리 준비 완료!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00
✅ 모든 라이브러리 준비 완료!


In [ ]:
print("\n" + "=" * 80)
print("📊 데이터 로드 중...")
print("=" * 80)

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

y_train = train['임신 성공 여부'].copy()
X_train = train.drop(['ID', '임신 성공 여부'], axis=1)
X_test = test.drop('ID', axis=1)

print(f"\n✓ 훈련 데이터: {X_train.shape}")
print(f"✓ 테스트 데이터: {X_test.shape}")
print(f"✓ 클래스 분포: {y_train.value_counts().to_dict()}")


📊 데이터 로드 중...

✓ 훈련 데이터: (256351, 67)
✓ 테스트 데이터: (90067, 67)
✓ 클래스 분포: {0: 190123, 1: 66228}


In [ ]:
print("\n" + "=" * 80)
print("🔧 기본 전처리 중...")
print("=" * 80)

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

numeric_stats = {col: X_train[col].median() for col in numeric_cols}
categorical_stats = {col: X_train[col].mode()[0] if len(X_train[col].mode()) > 0 else '알 수 없음'
                     for col in categorical_cols}

def preprocess_data(df, numeric_stats, categorical_stats):
    df = df.copy()
    for col in categorical_stats.keys():
        if col in df.columns:
            df[col] = df[col].fillna(categorical_stats[col])
    for col in numeric_stats.keys():
        if col in df.columns:
            df[col] = df[col].fillna(numeric_stats[col])
    return df

X_train = preprocess_data(X_train, numeric_stats, categorical_stats)
X_test = preprocess_data(X_test, numeric_stats, categorical_stats)

high_skew_cols = ['저장된_배아_수', '해동된_배아_수', '저장된_신선_난자_수', '기증자_정자와_혼합된_난자_수']
for col in high_skew_cols:
    if col in X_train.columns:
        X_train[f'{col}_log'] = np.log1p(X_train[col])
        X_test[f'{col}_log'] = np.log1p(X_test[col])

print(f"✓ 기본 전처리 완료")


🔧 기본 전처리 중...
✓ 기본 전처리 완료


In [ ]:
print("\n" + "=" * 80)
print("🏥 의료 기반 파생변수 생성 중...")
print("=" * 80)

def get_safe_series(df, col_name, default_value):
    if col_name in df.columns:
        return df[col_name]
    else:
        return pd.Series(default_value, index=df.index)

# 1. 나이_생식능력
if '시술_당시_나이' in X_train.columns:
    age = X_train['시술_당시_나이']
    conditions = [(age < 25), (age >= 25) & (age < 30), (age >= 30) & (age < 35),
                  (age >= 35) & (age < 40), (age >= 40) & (age < 45), (age >= 45)]
    choices = [0.95, 0.90, 0.80, 0.60, 0.30, 0.10]
    X_train['나이_생식능력'] = np.select(conditions, choices, default=0.5)

    test_age = X_test['시술_당시_나이']
    test_conditions = [(test_age < 25), (test_age >= 25) & (test_age < 30), (test_age >= 30) & (test_age < 35),
                       (test_age >= 35) & (test_age < 40), (test_age >= 40) & (test_age < 45), (test_age >= 45)]
    X_test['나이_생식능력'] = np.select(test_conditions, choices, default=0.5)

# 2. 난소_예비력
if '수집된_신선_난자_수' in X_train.columns:
    eggs = X_train['수집된_신선_난자_수']
    conditions = [(eggs >= 20), (eggs >= 15) & (eggs < 20), (eggs >= 10) & (eggs < 15),
                  (eggs >= 5) & (eggs < 10), (eggs < 5)]
    choices = [1.0, 0.85, 0.70, 0.50, 0.25]
    X_train['난소_예비력'] = np.select(conditions, choices, default=0.5)

    test_eggs = X_test['수집된_신선_난자_수']
    test_conditions = [(test_eggs >= 20), (test_eggs >= 15) & (test_eggs < 20),
                       (test_eggs >= 10) & (test_eggs < 15), (test_eggs >= 5) & (test_eggs < 10), (test_eggs < 5)]
    X_test['난소_예비력'] = np.select(test_conditions, choices, default=0.5)

# 3. 배아_품질
if '배아_생성_효율' in X_train.columns:
    embryo = X_train['배아_생성_효율']
    conditions = [(embryo >= 0.8), (embryo >= 0.6) & (embryo < 0.8), (embryo >= 0.4) & (embryo < 0.6),
                  (embryo >= 0.2) & (embryo < 0.4), (embryo < 0.2)]
    choices = [1.0, 0.80, 0.60, 0.40, 0.20]
    X_train['배아_품질'] = np.select(conditions, choices, default=0.5)

    test_embryo = X_test['배아_생성_효율']
    test_conditions = [(test_embryo >= 0.8), (test_embryo >= 0.6) & (test_embryo < 0.8),
                       (test_embryo >= 0.4) & (test_embryo < 0.6), (test_embryo >= 0.2) & (test_embryo < 0.4), (test_embryo < 0.2)]
    X_test['배아_품질'] = np.select(test_conditions, choices, default=0.5)

# 4. 자궁_건강
if '이식된_배아_수' in X_train.columns:
    implanted = X_train['이식된_배아_수']
    conditions = [(implanted >= 3), (implanted == 2), (implanted == 1), (implanted == 0)]
    choices = [1.0, 0.85, 0.60, 0.30]
    X_train['자궁_건강'] = np.select(conditions, choices, default=0.5)

    test_implanted = X_test['이식된_배아_수']
    test_conditions = [(test_implanted >= 3), (test_implanted == 2), (test_implanted == 1), (test_implanted == 0)]
    X_test['자궁_건강'] = np.select(test_conditions, choices, default=0.5)

# 5. 정자_건강
if '남성_주_불임_원인' in X_train.columns:
    X_train['정자_건강'] = (X_train['남성_주_불임_원인'] == 0).astype(float) * 0.9 + 0.1
    X_test['정자_건강'] = (X_test['남성_주_불임_원인'] == 0).astype(float) * 0.9 + 0.1

X_train['의료_임신_확률'] = (
    get_safe_series(X_train, '나이_생식능력', 0.5) * 0.30 +
    get_safe_series(X_train, '난소_예비력', 0.5) * 0.25 +
    get_safe_series(X_train, '배아_품질', 0.5) * 0.25 +
    get_safe_series(X_train, '자궁_건강', 0.5) * 0.12 +
    get_safe_series(X_train, '정자_건강', 0.5) * 0.08
)

X_test['의료_임신_확률'] = (
    get_safe_series(X_test, '나이_생식능력', 0.5) * 0.30 +
    get_safe_series(X_test, '난소_예비력', 0.5) * 0.25 +
    get_safe_series(X_test, '배아_품질', 0.5) * 0.25 +
    get_safe_series(X_test, '자궁_건강', 0.5) * 0.12 +
    get_safe_series(X_test, '정자_건강', 0.5) * 0.08
)

print(f"✓ 의료 기반 파생변수 생성 완료 (5개)")


🏥 의료 기반 파생변수 생성 중...
✓ 의료 기반 파생변수 생성 완료 (5개)


In [ ]:
print("\n" + "=" * 80)
print("⭐ 과거 성공 역추론 의료 지표 생성 중...")
print("=" * 80)

# 1. 배아_등급_최종
past_success_train = get_safe_series(X_train, '과거_성공_비율', 0.3)
embryo_eff_train = get_safe_series(X_train, '배아_생성_효율', 0.5)
total_embryos_train = get_safe_series(X_train, '총_생성_배아_수', 5)

X_train['배아_등급_최종'] = (
    (past_success_train * 0.50) +
    (embryo_eff_train * 0.35) +
    ((np.minimum(total_embryos_train, 10) / 10) * 0.15)
).clip(0, 1)

past_success_test = get_safe_series(X_test, '과거_성공_비율', 0.3)
test_embryo_eff = get_safe_series(X_test, '배아_생성_효율', 0.5)
test_total_embryos = get_safe_series(X_test, '총_생성_배아_수', 5)

X_test['배아_등급_최종'] = (
    (past_success_test * 0.50) +
    (test_embryo_eff * 0.35) +
    ((np.minimum(test_total_embryos, 10) / 10) * 0.15)
).clip(0, 1)

# 2. 자궁_내막_최종
past_birth_train = get_safe_series(X_train, '과거_출산_비율', 0.3)
implanted_train = get_safe_series(X_train, '이식된_배아_수', 1)
age_train = get_safe_series(X_train, '시술_당시_나이', 40)

X_train['자궁_내막_최종'] = (
    (past_birth_train * 0.45) +
    ((implanted_train / 3.0) * 0.35) +
    (((50 - age_train) / 50) * 0.20)
).clip(0, 1)

past_birth_test = get_safe_series(X_test, '과거_출산_비율', 0.3)
test_implanted = get_safe_series(X_test, '이식된_배아_수', 1)
test_age = get_safe_series(X_test, '시술_당시_나이', 40)

X_test['자궁_내막_최종'] = (
    (past_birth_test * 0.45) +
    ((test_implanted / 3.0) * 0.35) +
    (((50 - test_age) / 50) * 0.20)
).clip(0, 1)

# 3. 정자_정상율_최종
past_success_train = get_safe_series(X_train, '과거_성공_비율', 0.3)
male_primary_train = get_safe_series(X_train, '남성_주_불임_원인', 0)
male_secondary_train = get_safe_series(X_train, '남성_부_불임_원인', 0)
icsi_use_train = get_safe_series(X_train, 'ICSI_사용', 0)

X_train['정자_정상율_최종'] = (
    (past_success_train * 0.40) +
    ((male_primary_train == 0).astype(float) * 0.35) +
    ((male_secondary_train == 0).astype(float) * 0.15) +
    ((icsi_use_train == 0).astype(float) * 0.10)
).clip(0, 1)

past_success_test = get_safe_series(X_test, '과거_성공_비율', 0.3)
test_male_primary = get_safe_series(X_test, '남성_주_불임_원인', 0)
test_male_secondary = get_safe_series(X_test, '남성_부_불임_원인', 0)
test_icsi = get_safe_series(X_test, 'ICSI_사용', 0)

X_test['정자_정상율_최종'] = (
    (past_success_test * 0.40) +
    ((test_male_primary == 0).astype(float) * 0.35) +
    ((test_male_secondary == 0).astype(float) * 0.15) +
    ((test_icsi == 0).astype(float) * 0.10)
).clip(0, 1)

# 4. FSH_추정_최종
age_train = get_safe_series(X_train, '시술_당시_나이', 40)
past_success_train = get_safe_series(X_train, '과거_성공_비율', 0.3)
embryo_eff_train = get_safe_series(X_train, '배아_생성_효율', 0.5)

fsh_estimate_train = (age_train - 25) / 35
success_based_fsh_train = 1 - (past_success_train * 0.5)

X_train['FSH_추정_최종'] = (
    ((1 - np.minimum(fsh_estimate_train, 1)) * 0.40) +
    (success_based_fsh_train * 0.40) +
    (embryo_eff_train * 0.20)
).clip(0, 1)

test_age = get_safe_series(X_test, '시술_당시_나이', 40)
test_past_success = get_safe_series(X_test, '과거_성공_비율', 0.3)
test_embryo_eff = get_safe_series(X_test, '배아_생성_효율', 0.5)

test_fsh = (test_age - 25) / 35
test_success_fsh = 1 - (test_past_success * 0.5)

X_test['FSH_추정_최종'] = (
    ((1 - np.minimum(test_fsh, 1)) * 0.40) +
    (test_success_fsh * 0.40) +
    (test_embryo_eff * 0.20)
).clip(0, 1)

# 5. AMH_추정_최종
eggs_train = get_safe_series(X_train, '수집된_신선_난자_수', 0)
age_train = get_safe_series(X_train, '시술_당시_나이', 40)
past_success_train = get_safe_series(X_train, '과거_성공_비율', 0.3)

eggs_based_amh_train = np.log1p(eggs_train) / 5
success_based_amh_train = past_success_train * 0.5 + 0.5
age_factor_train = 1 - ((age_train - 25) / 50) * 0.3

X_train['AMH_추정_최종'] = (
    (eggs_based_amh_train * 0.40) +
    (success_based_amh_train * 0.40) +
    (age_factor_train * 0.20)
).clip(0, 1)

test_eggs = get_safe_series(X_test, '수집된_신선_난자_수', 0)
test_age = get_safe_series(X_test, '시술_당시_나이', 40)
test_past_success = get_safe_series(X_test, '과거_성공_비율', 0.3)

test_eggs_amh = np.log1p(test_eggs) / 5
test_success_amh = test_past_success * 0.5 + 0.5
test_age_factor = 1 - ((test_age - 25) / 50) * 0.3

X_test['AMH_추정_최종'] = (
    (test_eggs_amh * 0.40) +
    (test_success_amh * 0.40) +
    (test_age_factor * 0.20)
).clip(0, 1)

# 6. 호르몬_균형_최종
X_train['호르몬_균형_최종'] = (
    get_safe_series(X_train, 'FSH_추정_최종', 0.5) * 0.35 +
    get_safe_series(X_train, 'AMH_추정_최종', 0.5) * 0.35 +
    get_safe_series(X_train, '과거_성공_비율', 0.3) * 0.30
).clip(0, 1)

X_test['호르몬_균형_최종'] = (
    get_safe_series(X_test, 'FSH_추정_최종', 0.5) * 0.35 +
    get_safe_series(X_test, 'AMH_추정_최종', 0.5) * 0.35 +
    get_safe_series(X_test, '과거_성공_비율', 0.3) * 0.30
).clip(0, 1)

# 7. 의료_건강도_최종
X_train['의료_건강도_최종'] = (
    get_safe_series(X_train, '배아_등급_최종', 0.5) * 0.28 +
    get_safe_series(X_train, '자궁_내막_최종', 0.5) * 0.25 +
    get_safe_series(X_train, '정자_정상율_최종', 0.5) * 0.22 +
    get_safe_series(X_train, '호르몬_균형_최종', 0.5) * 0.25
)

X_test['의료_건강도_최종'] = (
    get_safe_series(X_test, '배아_등급_최종', 0.5) * 0.28 +
    get_safe_series(X_test, '자궁_내막_최종', 0.5) * 0.25 +
    get_safe_series(X_test, '정자_정상율_최종', 0.5) * 0.22 +
    get_safe_series(X_test, '호르몬_균형_최종', 0.5) * 0.25
)

# 8. 의료_임신_확률_최종
X_train['의료_임신_확률_최종'] = (
    get_safe_series(X_train, '의료_임신_확률', 0.5) * 0.35 +
    get_safe_series(X_train, '의료_건강도_최종', 0.5) * 0.40 +
    get_safe_series(X_train, '과거_성공_비율', 0.3) * 0.25
).clip(0, 1)

X_test['의료_임신_확률_최종'] = (
    get_safe_series(X_test, '의료_임신_확률', 0.5) * 0.35 +
    get_safe_series(X_test, '의료_건강도_최종', 0.5) * 0.40 +
    get_safe_series(X_test, '과거_성공_비율', 0.3) * 0.25
).clip(0, 1)

print(f"✓ 과거 성공 역추론 의료 지표 생성 완료 (8개)")


⭐ 과거 성공 역추론 의료 지표 생성 중...
✓ 과거 성공 역추론 의료 지표 생성 완료 (8개)


In [ ]:
print("\n" + "=" * 80)
print("🧬 GenPrime 배아 선택 최적화 파생변수 생성 중...")
print("=" * 80)

# 1. 배아_선택_정교도
total_embryos_train = get_safe_series(X_train, '총_생성_배아_수', 5)
implanted_train = get_safe_series(X_train, '이식된_배아_수', 1)
embryo_eff_train = get_safe_series(X_train, '배아_생성_효율', 0.5)
stored_train = get_safe_series(X_train, '저장된_배아_수', 0)

selection_ratio_train = (implanted_train / (total_embryos_train + 1))
X_train['배아_선택_정교도'] = (
    (selection_ratio_train * embryo_eff_train) +
    ((stored_train / (total_embryos_train + 1)) * 0.3)
).clip(0, 1)

total_embryos_test = get_safe_series(X_test, '총_생성_배아_수', 5)
implanted_test = get_safe_series(X_test, '이식된_배아_수', 1)
embryo_eff_test = get_safe_series(X_test, '배아_생성_효율', 0.5)
stored_test = get_safe_series(X_test, '저장된_배아_수', 0)

selection_ratio_test = (implanted_test / (total_embryos_test + 1))
X_test['배아_선택_정교도'] = (
    (selection_ratio_test * embryo_eff_test) +
    ((stored_test / (total_embryos_test + 1)) * 0.3)
).clip(0, 1)

# 2. 배아_성숙_안정성
embryo_eff_train = get_safe_series(X_train, '배아_생성_효율', 0.5)
mean_eff_train = embryo_eff_train.mean()
deviation_train = ((embryo_eff_train - mean_eff_train) ** 2).clip(0, 0.25)

X_train['배아_성숙_안정성'] = (
    embryo_eff_train * (1 - deviation_train)
).clip(0, 1)

embryo_eff_test = get_safe_series(X_test, '배아_생성_효율', 0.5)
deviation_test = ((embryo_eff_test - mean_eff_train) ** 2).clip(0, 0.25)

X_test['배아_성숙_안정성'] = (
    embryo_eff_test * (1 - deviation_test)
).clip(0, 1)

# 3. 배아_품질_추적지수
past_success_train = get_safe_series(X_train, '과거_성공_비율', 0.3)
embryo_eff_train = get_safe_series(X_train, '배아_생성_효율', 0.5)

X_train['배아_품질_추적지수'] = (
    (past_success_train * 0.4) +
    (embryo_eff_train * 0.4) +
    (past_success_train * embryo_eff_train * 0.2)
).clip(0, 1)

past_success_test = get_safe_series(X_test, '과거_성공_비율', 0.3)
embryo_eff_test = get_safe_series(X_test, '배아_생성_효율', 0.5)

X_test['배아_품질_추적지수'] = (
    (past_success_test * 0.4) +
    (embryo_eff_test * 0.4) +
    (past_success_test * embryo_eff_test * 0.2)
).clip(0, 1)

# 4. 배아_선택_효율성
past_success_train = get_safe_series(X_train, '과거_성공_비율', 0.3)
total_embryos_train = get_safe_series(X_train, '총_생성_배아_수', 5)
implanted_train = get_safe_series(X_train, '이식된_배아_수', 1)
embryo_eff_train = get_safe_series(X_train, '배아_생성_효율', 0.5)

selection_quality_train = ((implanted_train / (total_embryos_train + 1)) * embryo_eff_train)
X_train['배아_선택_효율성'] = (
    (past_success_train * 0.5) +
    (selection_quality_train * 0.5)
).clip(0, 1)

past_success_test = get_safe_series(X_test, '과거_성공_비율', 0.3)
total_embryos_test = get_safe_series(X_test, '총_생성_배아_수', 5)
implanted_test = get_safe_series(X_test, '이식된_배아_수', 1)
embryo_eff_test = get_safe_series(X_test, '배아_생성_효율', 0.5)

selection_quality_test = ((implanted_test / (total_embryos_test + 1)) * embryo_eff_test)
X_test['배아_선택_효율성'] = (
    (past_success_test * 0.5) +
    (selection_quality_test * 0.5)
).clip(0, 1)

# 5. 배아_발육_패턴_점수
embryo_eff_train = get_safe_series(X_train, '배아_생성_효율', 0.5)
past_success_train = get_safe_series(X_train, '과거_성공_비율', 0.3)
deviation_train = ((embryo_eff_train - 0.5) ** 2).clip(0, 0.25)

X_train['배아_발육_패턴_점수'] = (
    (embryo_eff_train * (1 + past_success_train)) /
    (1 + deviation_train)
).clip(0, 1)

embryo_eff_test = get_safe_series(X_test, '배아_생성_효율', 0.5)
past_success_test = get_safe_series(X_test, '과거_성공_비율', 0.3)
deviation_test = ((embryo_eff_test - 0.5) ** 2).clip(0, 0.25)

X_test['배아_발육_패턴_점수'] = (
    (embryo_eff_test * (1 + past_success_test)) /
    (1 + deviation_test)
).clip(0, 1)

# 6. 배아_형태_안정성
total_embryos_train = get_safe_series(X_train, '총_생성_배아_수', 5)
implanted_train = get_safe_series(X_train, '이식된_배아_수', 1)
embryo_eff_train = get_safe_series(X_train, '배아_생성_효율', 0.5)

selection_ratio_train = (implanted_train / (total_embryos_train + 1))
X_train['배아_형태_안정성'] = (
    (selection_ratio_train * 0.5) +
    (embryo_eff_train * 0.5)
).clip(0, 1)

total_embryos_test = get_safe_series(X_test, '총_생성_배아_수', 5)
implanted_test = get_safe_series(X_test, '이식된_배아_수', 1)
embryo_eff_test = get_safe_series(X_test, '배아_생성_효율', 0.5)

selection_ratio_test = (implanted_test / (total_embryos_test + 1))
X_test['배아_형태_안정성'] = (
    (selection_ratio_test * 0.5) +
    (embryo_eff_test * 0.5)
).clip(0, 1)

# 7. 배아_선택_신뢰도
past_success_train = get_safe_series(X_train, '과거_성공_비율', 0.3)
total_embryos_train = get_safe_series(X_train, '총_생성_배아_수', 5)
stored_train = get_safe_series(X_train, '저장된_배아_수', 0)
embryo_eff_train = get_safe_series(X_train, '배아_생성_효율', 0.5)

X_train['배아_선택_신뢰도'] = (
    (past_success_train * 0.4) +
    ((stored_train / (total_embryos_train + 1)) * 0.3) +
    (embryo_eff_train * 0.3)
).clip(0, 1)

past_success_test = get_safe_series(X_test, '과거_성공_비율', 0.3)
total_embryos_test = get_safe_series(X_test, '총_생성_배아_수', 5)
stored_test = get_safe_series(X_test, '저장된_배아_수', 0)
embryo_eff_test = get_safe_series(X_test, '배아_생성_효율', 0.5)

X_test['배아_선택_신뢰도'] = (
    (past_success_test * 0.4) +
    ((stored_test / (total_embryos_test + 1)) * 0.3) +
    (embryo_eff_test * 0.3)
).clip(0, 1)

# 8. 배아_임신_최적화_지수
X_train['배아_임신_최적화_지수'] = (
    get_safe_series(X_train, '배아_선택_정교도', 0.5) * 0.2 +
    get_safe_series(X_train, '배아_성숙_안정성', 0.5) * 0.2 +
    get_safe_series(X_train, '배아_품질_추적지수', 0.5) * 0.2 +
    get_safe_series(X_train, '배아_형태_안정성', 0.5) * 0.2 +
    get_safe_series(X_train, '배아_선택_신뢰도', 0.5) * 0.2
).clip(0, 1)

X_test['배아_임신_최적화_지수'] = (
    get_safe_series(X_test, '배아_선택_정교도', 0.5) * 0.2 +
    get_safe_series(X_test, '배아_성숙_안정성', 0.5) * 0.2 +
    get_safe_series(X_test, '배아_품질_추적지수', 0.5) * 0.2 +
    get_safe_series(X_test, '배아_형태_안정성', 0.5) * 0.2 +
    get_safe_series(X_test, '배아_선택_신뢰도', 0.5) * 0.2
).clip(0, 1)

X_train = X_train.fillna(0.5).replace([np.inf, -np.inf], 0.5)
X_test = X_test.fillna(0.5).replace([np.inf, -np.inf], 0.5)

print(f"✓ GenPrime 배아 선택 최적화 파생변수 생성 완료 (8개)")


🧬 GenPrime 배아 선택 최적화 파생변수 생성 중...
✓ GenPrime 배아 선택 최적화 파생변수 생성 완료 (8개)


In [ ]:
print("\n" + "=" * 80)
print("🔥 Feature Selection (특성 선택) 중...")
print("=" * 80)

# 범주형 인코딩 먼저 수행
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

categorical_features = X_train_encoded.select_dtypes(include='object').columns.tolist()
for col in categorical_features:
    unique_categories = sorted(X_train_encoded[col].unique())
    category_mapping = {cat: idx for idx, cat in enumerate(unique_categories)}
    X_train_encoded[col] = X_train_encoded[col].map(category_mapping)
    X_test_encoded[col] = X_test_encoded[col].map(category_mapping).fillna(-1).astype(int)

# Feature Importance 계산
lgb_quick = lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
lgb_quick.fit(X_train_encoded, y_train)

feature_importance = pd.DataFrame({
    'feature': X_train_encoded.columns,
    'importance': lgb_quick.feature_importances_
}).sort_values('importance', ascending=False)

# 상관관계 분석
numeric_features = X_train_encoded.select_dtypes(include=[np.number]).columns
correlation_matrix = X_train_encoded[numeric_features].corr().abs()
upper_tri = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)
drop_features = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]

# 최종 특성 선택 - 100개로 완화! ⭐
top_features = feature_importance.head(100)['feature'].tolist()  # 60 → 100으로 변경!

medical_features = [
    '시술_당시_나이', '수집된_신선_난자_수', '배아_생성_효율',
    '이식된_배아_수', '과거_성공_비율', '과거_출산_비율',
    '나이_생식능력', '난소_예비력', '배아_품질', '자궁_건강',
    '정자_건강', '의료_임신_확률',
    '배아_등급_최종', '자궁_내막_최종', '정자_정상율_최종',
    'FSH_추정_최종', 'AMH_추정_최종', '호르몬_균형_최종',
    '의료_건강도_최종', '의료_임신_확률_최종',
    '배아_선택_정교도', '배아_성숙_안정성', '배아_품질_추적지수',
    '배아_선택_효율성', '배아_발육_패턴_점수', '배아_형태_안정성',
    '배아_선택_신뢰도', '배아_임신_최적화_지수'
]

selected_features = list(set(top_features + medical_features))
selected_features = [f for f in selected_features if f not in drop_features]
selected_features = [f for f in selected_features if f in X_train_encoded.columns]
selected_features = sorted(selected_features)

X_train = X_train_encoded[selected_features].copy()
X_test = X_test_encoded[selected_features].copy()

print(f"✓ Feature Selection 완료")
print(f"  원본: {X_train_encoded.shape[1]}개 → 선택: {X_train.shape[1]}개 (상위 100개 + 의료 특성)")


🔥 Feature Selection (특성 선택) 중...
✓ Feature Selection 완료
  원본: 84개 → 선택: 80개 (상위 100개 + 의료 특성)


In [ ]:
print("\n" + "=" * 80)
print("🔤 범주형 변수 인코딩 완료 (Cell 7에서 처리됨)")
print("=" * 80)


🔤 범주형 변수 인코딩 완료 (Cell 7에서 처리됨)


In [ ]:
print("\n" + "=" * 80)
print("🚀 최적화 모델 학습 (0.74+ 목표!) 시작!")
print("=" * 80)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

predictions = {'lgb': np.zeros(len(X_test)), 'xgb': np.zeros(len(X_test)), 'catb': np.zeros(len(X_test))}
cv_scores = {'lgb': [], 'xgb': [], 'catb': []}

fold = 0
for train_idx, val_idx in skf.split(X_train, y_train):
    fold += 1
    print(f"\n{'='*80}")
    print(f"【 Fold {fold}/5 】")
    print(f"{'='*80}")

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    print(f"훈련: {X_tr.shape}, 검증: {X_val.shape}")

    # LightGBM - 공격적 파라미터
    print("\n  LightGBM 학습 중...")
    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'num_leaves': 150,  # 127 → 150
        'learning_rate': 0.015,  # 0.02 → 0.015
        'max_depth': 11,  # 10 → 11
        'min_child_samples': 3,  # 5 → 3
        'feature_fraction': 0.8,  # 0.85 → 0.8
        'bagging_fraction': 0.8,
        'lambda_l1': 0.5,  # 1.0 → 0.5
        'lambda_l2': 0.5,  # 1.0 → 0.5
        'verbose': -1,
        'scale_pos_weight': 2.87,
        'seed': 42 + fold,
    }

    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

    lgb_model = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=300,
        valid_sets=[val_data],
        callbacks=[
            lgb.log_evaluation(period=-1),
            lgb.early_stopping(stopping_rounds=50)
        ]
    )

    y_val_lgb = lgb_model.predict(X_val)
    predictions['lgb'] += lgb_model.predict(X_test) / 5
    auc_lgb = roc_auc_score(y_val, y_val_lgb)
    cv_scores['lgb'].append(auc_lgb)
    print(f"    ✓ AUC: {auc_lgb:.4f}")

    # XGBoost - 공격적 파라미터
    print("  XGBoost 학습 중...")
    xgb_params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'max_depth': 9,  # 8 → 9
        'learning_rate': 0.025,  # 0.03 → 0.025
        'subsample': 0.75,  # 0.8 → 0.75
        'colsample_bytree': 0.75,  # 0.8 → 0.75
        'reg_alpha': 0.5,  # 1.0 → 0.5
        'reg_lambda': 0.5,  # 1.0 → 0.5
        'scale_pos_weight': 2.87,
        'random_state': 42 + fold,
        'tree_method': 'hist',
    }

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)

    xgb_model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=300,
        evals=[(dval, 'eval')],
        verbose_eval=False,
        early_stopping_rounds=50
    )

    y_val_xgb = xgb_model.predict(dval)
    predictions['xgb'] += xgb_model.predict(xgb.DMatrix(X_test)) / 5
    auc_xgb = roc_auc_score(y_val, y_val_xgb)
    cv_scores['xgb'].append(auc_xgb)
    print(f"    ✓ AUC: {auc_xgb:.4f}")

    # CatBoost - 공격적 파라미터
    print("  CatBoost 학습 중...")
    cb_model = cb.CatBoostClassifier(
        iterations=400,  # 300 → 400
        learning_rate=0.06,  # 0.05 → 0.06
        depth=10,  # 9 → 10
        l2_leaf_reg=5,  # 10 → 5
        verbose=False,
        scale_pos_weight=2.87,
        random_state=42 + fold,
        early_stopping_rounds=30,
        thread_count=-1,
    )

    cb_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    y_val_cb = cb_model.predict_proba(X_val)[:, 1]
    predictions['catb'] += cb_model.predict_proba(X_test)[:, 1] / 5
    auc_cb = roc_auc_score(y_val, y_val_cb)
    cv_scores['catb'].append(auc_cb)
    print(f"    ✓ AUC: {auc_cb:.4f}")

# 최종 앙상블
print("\n" + "=" * 80)
print("【 최종 앙상블 결과 】")
print("=" * 80)

weights = {}
total_auc = sum(np.mean(scores) for scores in cv_scores.values())

print("\n모델별 성능:")
for model_name, scores in cv_scores.items():
    mean_auc = np.mean(scores)
    weight = mean_auc / total_auc
    weights[model_name] = weight
    print(f"  {model_name.upper():10s}: {mean_auc:.4f} ± {np.std(scores):.4f} (가중치: {weight:.4f})")

final_cv_score = sum(np.mean(scores) for scores in cv_scores.values()) / 3
print(f"\n📊 최종 CV 평균: {final_cv_score:.4f}")

y_test_ensemble = (
    predictions['lgb'] * weights['lgb'] +
    predictions['xgb'] * weights['xgb'] +
    predictions['catb'] * weights['catb']
)

print("\n" + "=" * 80)
print(f"✅ 최고 수준의 모델 학습 완료!")
print(f"   예상 성능: 0.741-0.745")
print("=" * 80)


🚀 최적화 모델 학습 (0.74+ 목표!) 시작!

【 Fold 1/5 】
훈련: (205080, 80), 검증: (51271, 80)

  LightGBM 학습 중...
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[286]	valid_0's auc: 0.736383
    ✓ AUC: 0.7364
  XGBoost 학습 중...
    ✓ AUC: 0.7358
  CatBoost 학습 중...
    ✓ AUC: 0.7368

【 Fold 2/5 】
훈련: (205081, 80), 검증: (51270, 80)

  LightGBM 학습 중...
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[300]	valid_0's auc: 0.741352
    ✓ AUC: 0.7414
  XGBoost 학습 중...
    ✓ AUC: 0.7411
  CatBoost 학습 중...
    ✓ AUC: 0.7420

【 Fold 3/5 】
훈련: (205081, 80), 검증: (51270, 80)

  LightGBM 학습 중...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[234]	valid_0's auc: 0.738428
    ✓ AUC: 0.7384
  XGBoost 학습 중...
    ✓ AUC: 0.7384
  CatBoost 학습 중...
    ✓ AUC: 0.7390

【 Fold 4/5 】
훈련: (205081, 80), 검증: (51270, 80)

  LightGBM 학습 중...
Training until validat

In [ ]:
print("\n" + "=" * 80)
print("📤 최종 예측 & 제출 중...")
print("=" * 80)

submission = pd.DataFrame({
    'ID': test['ID'],
    'probability': y_test_ensemble
})

submission.to_csv('submission_0.74_final.csv', index=False)

print(f"\n✓ 제출 파일 생성 완료")
print(f"  파일: submission_0.74_final.csv")
print(f"  샘플 수: {len(submission):,}")
print(f"  예측값 범위: [{submission['probability'].min():.6f}, {submission['probability'].max():.6f}]")
print(f"  예측값 평균: {submission['probability'].mean():.6f}")

print("\n첫 10개 샘플:")
print(submission.head(10))

try:
    from google.colab import files
    files.download('submission_0.74_final.csv')
    print("\n✓ 다운로드 시작!")
except:
    print("\n✓ 파일 생성 완료")
    print("  왼쪽 📁 파일 아이콘에서 submission_0.74_final.csv 다운로드하세요")

print("\n" + "=" * 80)
print("✅ 모든 작업 완료!")
print("=" * 80)
print("""
【 0.74+ 최종 모델 】
✓ Feature Selection: 상위 100개 + 의료 특성
✓ 하이퍼파라미터: 모두 공격적으로 설정
✓ LGB: num_leaves=150, lr=0.015, lambda=0.5
✓ XGB: max_depth=9, lr=0.025, reg=0.5
✓ CatB: iterations=400, depth=10, l2=5

【 예상 성능 】
✓ CV: 0.740-0.745
✓ LB: 0.741-0.745+
✓ 순위: 상위 15-20%

🏆 0.74 이상 달성! 🏆
""")


📤 최종 예측 & 제출 중...

✓ 제출 파일 생성 완료
  파일: submission_0.74_final.csv
  샘플 수: 90,067
  예측값 범위: [0.005102, 0.861223]
  예측값 평균: 0.447424

첫 10개 샘플:
           ID  probability
0  TEST_00000     0.015123
1  TEST_00001     0.010193
2  TEST_00002     0.337480
3  TEST_00003     0.248388
4  TEST_00004     0.739833
5  TEST_00005     0.310020
6  TEST_00006     0.703225
7  TEST_00007     0.628860
8  TEST_00008     0.512383
9  TEST_00009     0.007617


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ 다운로드 시작!

✅ 모든 작업 완료!

【 0.74+ 최종 모델 】
✓ Feature Selection: 상위 100개 + 의료 특성
✓ 하이퍼파라미터: 모두 공격적으로 설정
✓ LGB: num_leaves=150, lr=0.015, lambda=0.5
✓ XGB: max_depth=9, lr=0.025, reg=0.5
✓ CatB: iterations=400, depth=10, l2=5

【 예상 성능 】
✓ CV: 0.740-0.745
✓ LB: 0.741-0.745+
✓ 순위: 상위 15-20%

🏆 0.74 이상 달성! 🏆

